In [8]:
import os
import glob
import zipfile

import gc

import pandas as pd

# Enable and run garbage collection to free memory before processing
gc.enable()
gc.collect()

3772

In [9]:
# Read in BV-BRC genome list (exported from https://www.bv-brc.org/view/Taxonomy/1579#view_tab=genomes)
df = pd.read_excel('BVBRC_genome.xlsx')
df = df[df['Assembly Accession'].notna()]

# Remove entries with multiple Assembly Accessions and keep only unique records
df = df[~df['Assembly Accession'].str.contains(',')].drop_duplicates(subset=['Assembly Accession'])

# Assume missing CheckM Contamination values are 0
df['CheckM Contamination'] = df['CheckM Contamination'].fillna(0)

# Retain only genomes with >90% completeness and <5% contamination
df = df[(df['CheckM Completeness'] > 90) & (df['CheckM Contamination'] < 5)]

df.head(1)

,Genome ID,Genome Name,Other Names,NCBI Taxon ID,Taxon Lineage IDs,Taxon Lineage Names,Superkingdom,Kingdom,Phylum,Class,...,Host Age,Host Health,Host Group,Lab Host,Passage,Other Clinical,Additional Metadata,Comments,Date Inserted,Date Modified
0,1226675.3,Lactobacillus acidophilus CIP 76.13,NaN,1423717,131567;2;1783272;1239;91061;186826;33958;1578;...,cellular organisms;Bacteria;Bacillati;Bacillot...,Bacteria,Bacillati,Bacillota,Bacilli,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The complete genome sequences of several pairs...,2014-12-08T22:11:52.067Z,2015-03-16T03:17:09.594Z


In [ ]:
### Note: These files have already been downloaded and are located in the project's data folder.

%mkdir -p tmp
%cd tmp

In [11]:
## Retrieve protein.faa files for each Assembly Accession and concatenate all into one combined proteins.faa file
## Create a mapping of each protein ID to its Assembly Accession, and also save protein length

# # Retrieve protein.faa files
# for accession in df['Assembly Accession'].unique():
#     try:
#         output = f"{accession}.zip"
#         if os.path.exists(output):
#             continue
#         !datasets download genome accession $accession --include protein --filename $output
#         time.sleep(5)
#     except Exception as e:
#         print(e)
#         raise

%ls

GCA_000011985.1.zip*  GCA_017009655.1.zip*  GCA_027680805.1.zip*
GCA_000159715.1.zip*  GCA_017009695.1.zip*  GCA_027684755.1.zip*
GCA_000191545.1.zip*  GCA_017009715.1.zip*  GCA_027687975.1.zip*
GCA_000389675.2.zip*  GCA_017009725.1.zip*  GCA_027689875.1.zip*
GCA_000442825.1.zip*  GCA_017695935.1.zip*  GCA_027690645.1.zip*
GCA_000442865.1.zip*  GCA_018252545.1.zip*  GCA_027690935.1.zip*
GCA_000469705.1.zip*  GCA_018367455.1.zip*  GCA_027692675.1.zip*
GCA_000469745.1.zip*  GCA_020883435.1.zip*  GCA_029334915.1.zip*
GCA_000469765.1.zip*  GCA_021229035.1.zip*  GCA_030369715.1.zip*
GCA_000497795.1.zip*  GCA_021432145.1.zip*  GCA_030520045.1.zip*
GCA_000786395.1.zip*  GCA_022509485.1.zip*  GCA_030520065.1.zip*
GCA_002224305.1.zip*  GCA_023093425.1.zip*  GCA_032463485.1.zip*
GCA_002286215.1.zip*  GCA_024397395.1.zip*  GCA_032917925.1.zip*
GCA_002406675.1.zip*  GCA_024665075.1.zip*  GCA_033569435.1.zip*
GCA_002914945.1.zip*  GCA_024665555.1.zip*  GCA_033598375.1.zip*
GCA_003053135.1.zip*  GCA

In [12]:
## Concatenate all into one combined proteins.faa file
## Create a mapping of each protein ID to its Assembly Accession, and also save protein length

combined_fasta = "proteins.faa"
mapping_file = "protein_to_genome.tsv"

def parse_faa(content):
    """Parse faa content, yielding (protein_id, sequence) tuples."""
    lines = content.decode("utf-8").splitlines()
    prot_id = None
    seq_parts = []

    for line in lines:
        if line.startswith(">"):
            if prot_id:
                yield prot_id, prot_id_full, "\n".join(seq_parts), len("".join(seq_parts))
            prot_id = line.split()[0].replace(">", "")
            prot_id_full = line.replace(">", "")
            seq_parts = []
        else:
            seq_parts.append(line.strip())

    if prot_id:
        yield prot_id, prot_id_full, "\n".join(seq_parts), len("".join(seq_parts))


with open(combined_fasta, "w") as out_faa, open(mapping_file, "w") as out_map:
    out_map.write("protein_id\tgenome_id\tprotein_length\n")

    for z_file in glob.glob("GC*_*.zip"):
        genome_id = z_file.replace(".zip", "")

        with zipfile.ZipFile(z_file, 'r') as z:
            protein_file = next(
                (f for f in z.namelist() if f.endswith("protein.faa")), None)
            if not protein_file:
                continue

            content = z.read(protein_file)

            for prot_id, prot_id_full, sequence, seqlen in parse_faa(content):
                out_faa.write(f">{prot_id_full}\n{sequence}\n")
                out_map.write(f"{prot_id}\t{genome_id}\t{seqlen}\n")

In [ ]:
%cd ..

In [14]:
mapping = pd.read_table('tmp/protein_to_genome.tsv')
mapping['Genome Name'] = mapping['genome_id'].map(df.set_index(['Assembly Accession'])['Genome Name'].to_dict())
mapping.head(2)

,protein_id,genome_id,protein_length,Genome Name
0,AAV41909.1,GCA_000011985.1,455,Lactobacillus acidophilus NCFM
1,AAV41910.1,GCA_000011985.1,376,Lactobacillus acidophilus NCFM


In [90]:
# Create a DIAMOND database from all L. acidophilus strain protein sequences for BLASTX alignment
!diamond makedb --in tmp/proteins.faa --db acidophilus_db

/bin/bash: line 1: diamond: command not found


In [8]:
FILES = glob.glob("../VZK_data/*/")  # Path to directories containing R1.fastq.gz and R2.fastq.gz
# Check path
FILES[0] # e.g. '../VZK_data/int7/'

'../VZK_data/int7/'

In [ ]:
%%capture cap


for FILE in FILES:

    _, _, ID, _ = FILE.split("/")


    print(f"working with {ID}")
    
    R1 = glob.glob(f"{FILE}**1.fq.gz")[0]
    R2 = glob.glob(f"{FILE}**2.fq.gz")[0]

    R1C = R1.replace(".fq.gz", "_clean.fq.gz")
    R2C = R2.replace(".fq.gz", "_clean.fq.gz")

    R12F = f"{FILE}classified_#c.fq"

    REPORT = f"{FILE}{ID}_fastp_report.html"
    
    ## Assess sequencing quality and perform initial read filtering
    print(f"fastp ...")
    !fastp -i $R1 -I $R2 -o $R1C -O $R2C -h $REPORT -w 16 -q 30 -l 35 -u 40
    print(f"complete\n")
    
    ## Remove potential host-derived and other contaminated reads
    print(f"hostile ...")
    
    # Align against the rat reference genome (GRCr8)
    !hostile clean --fastq1 $R1C --fastq2 $R2C  -o $FILE -t 12

    !rm  $R1C
    !rm  $R2C

    R1CD = R1C.replace('.fq.gz', ".clean_1.fastq.gz")
    R2CD = R2C.replace('.fq.gz', ".clean_2.fastq.gz")

    !hostile clean --fastq1 $R1CD --fastq2 $R2CD  -o $FILE -t 12 --index rattus_norvegicus_db/rattus_norvegicus

    !rm  $R1CD
    !rm  $R2CD

    R1CD = f"{FILE}{ID}_1_clean.clean_1.clean_1.fastq.gz"
    R2CD = f"{FILE}{ID}_2_clean.clean_2.clean_2.fastq.gz"

    print(f"complete\n")

    REPORT = f"{FILE}{ID}_alignment.tsv"
    
    ## Align reads against the L. acidophilus DIAMOND database
    print(f"diamond ...")
    !diamond blastx \
      --db acidophilus_db.dmnd \
      --query $R1CD \
      --out $REPORT \
      --outfmt 6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore \
      --threads 16 \
      -b 2  \
      -c 4  \
      -k 3 \
      --faster \
      --quiet \
      --max-target-seqs 1 \
      --min-score 50

    print(f"complete\n")
    
    ## If needed, remove temporary files to free up disk space
    !rm  $R1CD
    !rm  $R2CD
    
    gc.collect()

In [ ]:
# Cluster L. acidophilus strain protein sequences at 80% identity using MMseqs2
!mmseqs easy-cluster tmp/proteins.faa clusterRes tmp --min-seq-id 0.8 -c 0.8 --cov-mode 3 --threads 8

In [17]:
# Read the MMseqs2 clustering adjacency table
c = pd.read_table('clusterRes_cluster.tsv', header=None)
MMSEQ_CLUSTER_MAP = {}

# Create a mapping from each protein ID to its cluster ID and size
for cnt, row in enumerate(c.groupby(0)[1].apply(list).values):
    cluster_name = f"Cluster{cnt}-{len(row)}"
    for r in row:
        MMSEQ_CLUSTER_MAP[r] = cluster_name

# Map protein IDs to MMseqs2 clusters
mapping['mmseqs2-80%'] = mapping['protein_id'].map(MMSEQ_CLUSTER_MAP)
mapping = mapping.sort_values(by='mmseqs2-80%', ascending=True)

# Assign the most common NCBI protein naming to each cluster
MAP = {}
with open('tmp/proteins.faa', 'r') as f:
    for line in f:
        if line.startswith('>'):
            # Extract protein ID (first field after '>')
            ID = line[1:].split()[0]
            # Extract the protein name (everything after the ID, before ' [Lac')
            name = line[1 + len(ID):].split(' [Lac')[0].strip()
            MAP[ID] = name

# Map protein IDs to their NCBI annotations
mapping['annot'] = mapping['protein_id'].map(MAP)

# For each cluster, find the most frequent annotation
mapping2 = mapping.groupby('mmseqs2-80%')['annot'].value_counts().reset_index()
mapping2 = mapping2.drop_duplicates(subset='mmseqs2-80%')
most_common_annot = mapping2.set_index('mmseqs2-80%')['annot'].to_dict()
mapping['common_name'] = mapping['mmseqs2-80%'].map(most_common_annot)

# Display clustering and naming results
mapping.head(2)

,protein_id,genome_id,protein_length,Genome Name,mmseqs2-80%,annot,common_name
23618,PCL30934.1,GCA_002406675.1,376,Lactobacillus acidophilus strain P2,Cluster0-91,DNA polymerase III subunit beta,DNA polymerase III subunit beta
85625,MBO8212468.1,GCA_017695935.1,376,Lactobacillus acidophilus strain APC2845,Cluster0-91,DNA polymerase III subunit beta,DNA polymerase III subunit beta


In [18]:
meta = pd.read_excel('id-meta.xlsx', index_col='Sample')
meta.head()

,Group,Alias,microbiome sapmle ID
Sample,,,
VZK1-2,control,CON,int7
VZK1-4,control,CON,int8
VZK1-6,control,CON,int9
VZK2-1,control,CON,VZK2-1
VZK2-2,control,CON,VZK2-2


In [20]:
FILES = glob.glob("../VZK_data/*/*_alignment.tsv") # Path to DIAMOND alignment results
# Check path
FILES[0]

'../VZK_data/int7/int7_alignment.tsv'

In [21]:
RES = {}
RES2 = {}

M = mapping.set_index('protein_id')['common_name'].to_dict()
M2 = mapping.set_index('common_name')['protein_length'].to_dict()
M3 = mapping.set_index('protein_id')['genome_id'].to_dict()

for FILE in FILES:
    _, _, ID, _ = FILE.split("/")
    print(f"Working with {ID}")

    # Load DIAMOND alignment results
    align_cols = [
        "qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
        "qstart", "qend", "sstart", "send", "evalue", "bitscore"
    ]
    df = pd.read_csv(FILE, sep="\t", names=align_cols)

    # Filter for confident alignments (identity >= 80%, e-value <= 1e-5)
    df = df[(df['pident'] >= 80) & (df['evalue'] <= 1e-5)]

    df['common_name'] = df['sseqid'].map(M)
    df['origin_genome'] = df['sseqid'].map(M3)

    # Calculate the number of proteins covered by alignment in each reference genome
    protein_covered = pd.concat(
        (df.groupby('origin_genome')['sseqid'].nunique(),
         mapping.groupby('genome_id')['protein_id'].nunique()),
        axis=1).fillna(0)

    protein_covered.columns = ['covered_proteins', 'total_proteins']
    protein_covered['genome_coverage'] = protein_covered['covered_proteins'] / protein_covered['total_proteins'] * 100
    protein_covered['Genome Name'] = protein_covered.index.map(mapping.set_index('genome_id')['Genome Name'].to_dict())
    protein_covered = protein_covered.sort_values(by='genome_coverage', ascending=False)

    RES2[ID] = protein_covered.set_index('Genome Name')['genome_coverage'].to_dict()

    # Estimate relative abundance by dividing the number of times a protein was recovered by its length
    d = pd.DataFrame(df['common_name'].value_counts())
    d['protein_length'] = d.index.map(M2)
    d = (d['count'] / d['protein_length']).to_dict()

    RES[ID] = d

Working with int7
Working with int8
Working with int9
Working with VZK2-1
Working with VZK2-2
Working with VZK2-3
Working with VZK2-4
Working with VZK2-5
Working with VZK2-6
Working with VZK3-2
Working with VZK3-3
Working with VZK3-5
Working with VZK3-6
Working with VZK4-10
Working with VZK4-11
Working with VZK4-12
Working with VZK4-8
Working with VZK4-9
Working with VZK5-1
Working with VZK5-3
Working with VZK5-4
Working with VZK5-5
Working with VZK5-6
Working with VZK6-13
Working with VZK6-14
Working with VZK6-16
Working with VZK6-18


In [22]:
# Display estimated relative abundance of functional terms
RES = pd.DataFrame(RES).T.fillna(0)
RELAB = (RES.div(RES.sum(axis=1), axis=0) * 100)

RELAB = RELAB.loc[meta['microbiome sapmle ID']]
RELAB.index = meta.index

RELAB.head(2)

,DNA-directed RNA polymerase subunit beta,DNA-directed RNA polymerase subunit beta',S14 family endopeptidase Clp,elongation factor Tu,F0F1 ATP synthase subunit beta,Uncharacterised protein,transposase,amino acid permease,ABC transporter ATP-binding protein,excinuclease ABC subunit A,...,16S rRNA pseudouridylate synthase B,hypothetical protein LAC30SC_07840,phage-like protein,protein gp15,medium chain dehydrogenases/reductase (MDR)/zinc-dependent alcohol dehydrogenase-like family protein,hypothetical protein LAC30SC_10890 (plasmid),"glycosyl transferase, partial",Phosphoribosyl-ATP pyrophosphatase,RNA polymerase III,"ABC transporter, ATP-binding and permease protein"
Sample,,,,,,,,,,,,,,,,,,,,,
VZK1-2,0.591087,0.557845,0.645890,1.313985,0.827733,2.357863,5.488743,0.643914,0.540153,0.364100,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
VZK1-4,1.124689,1.204751,1.349363,2.815646,1.419241,1.842806,3.690894,0.409641,0.358324,0.685474,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
# Display genome coverage breadth, i.e., the number of unique proteins recovered per reference genome
RELAB2 = pd.DataFrame(RES2).T.fillna(0)
RELAB2 = RELAB2.loc[meta['microbiome sapmle ID']]
RELAB2.index = meta.index
RELAB2.head(2)

,Lactobacillus acidophilus strain NCTC13720,Lactobacillus acidophilus strain NCTC13721,Lactobacillus acidophilus 30SC,Lactobacillus acidophilus NCFM,Lactobacillus acidophilus strain NCTC1407,Lactobacillus acidophilus CIRM-BIA 444,Lactobacillus acidophilus ATCC 4796,Lactobacillus acidophilus strain YT1,Lactobacillus acidophilus La-14,Lactobacillus acidophilus DSM 20242,...,Lactobacillus acidophilus strain BCRC 80064,Lactobacillus acidophilus strain BCRC 16099,Lactobacillus acidophilus strain BCRC 12255,Lactobacillus acidophilus strain BIO6307,Lactobacillus acidophilus strain DS1_1A,Lactobacillus acidophilus strain DS8_1A,Lactobacillus acidophilus strain DS9_1A,Lactobacillus acidophilus strain DS10_1A,Lactobacillus acidophilus strain LA1,Lactobacillus acidophilus AM-LA-19
Sample,,,,,,,,,,,,,,,,,,,,,
VZK1-2,80.355221,56.145142,47.401651,45.918367,14.712286,11.476214,8.910891,8.311553,2.452026,2.076125,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
VZK1-4,73.573735,24.385486,29.480330,31.310419,12.255054,4.610103,5.445545,4.391009,1.066098,1.285220,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
# Save estimated stain data
RELAB.to_csv('prodata/la-relab.tsv.gz', sep='\t', compression='gzip')

# Save estimated stain coverage data
RELAB2.to_csv('prodata/la-coverage.tsv.gz', sep='\t', compression='gzip')